# LoRA Fine-Tune: Structured Extraction from Loan-Policy Text

**Kaggle GPU kernel. Status: v3 COMPLETE AND PASSED (2026-11-04).**

v3 fixed the proven v2 data bug (product was unlearnable: 0/300 v2 prompts named any product, 49 train policy texts mapped to multiple gold products). Every v3 policy template now names the product in the sentence itself. v3 ran on Kaggle kernel version 3 (~11 min, Tesla T4): BASE overall 0.0, LoRA overall 1.0 on the 60 held-out examples — notebook verdict PASS.

This kernel is self-contained. It generates synthetic data, fine-tunes Qwen2.5-0.5B-Instruct with LoRA, then compares base vs LoRA on held-out examples. The project is published at https://github.com/Tejasv609/lora-policy-extraction.


In [ ]:
# Kaggle only: install the training stack.
!pip install -q transformers>=4.42 peft>=0.12 accelerate>=0.33 datasets>=2.20 "torchao>=0.16"


In [ ]:
import os
import torch

print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
WORK = '/kaggle/working'
DATA = os.path.join(WORK, 'data')
os.makedirs(DATA, exist_ok=True)
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUT = os.path.join(WORK, 'outputs/lora-policy')

## 1. Synthetic dataset (no real data)

Templates only, fictional institution, gold labels known by construction. 240 train / 60 held-out, seed 42.

In [ ]:
import json, os, random
INSTITUTION = "Example Finance"  # fictional, synthetic-only
PRODUCTS = ["personal loan", "home loan", "auto loan", "small business loan", "education loan"]
DOC_SETS = [
    ["identity proof", "address proof", "income proof"],
    ["identity proof", "bank statements", "salary slips"],
    ["identity proof", "address proof", "property documents"],
    ["identity proof", "business registration", "bank statements"],
    ["identity proof", "admission letter", "fee structure"],
]
COLLATERAL = ["property", "vehicle", "fixed deposit", "gold", "none"]
FIELDS = {"eligibility": "min_credit_score", "interest_rate": "interest_rate_percent",
          "repayment": "max_tenure_months", "documentation": "required_documents",
          "collateral": "collateral_type"}

def sample_value(rng, category):
    if category == "eligibility":
        return rng.randrange(650, 801, 10)
    if category == "interest_rate":
        return round(rng.uniform(8.0, 14.5), 1)
    if category == "repayment":
        return rng.choice([12, 24, 36, 48, 60, 84, 120, 180, 240])
    if category == "documentation":
        return rng.choice(DOC_SETS)
    return rng.choice(COLLATERAL)

# v3 fix: every template NAMES the product in the sentence itself.
# v2 said only "this product", so product was unlearnable (0/300 v2 prompts
# named any product; 49 train texts mapped to multiple gold products).
# Do NOT revert to "this product", and do NOT add a "Product: X" header.
def templates(category, v, product):
    if category == "eligibility":
        return [f"Applicants for the {product} at {INSTITUTION} must have a minimum credit score of {v}.",
                f"To qualify for the {product}, the applicant needs a credit score of at least {v}."]
    if category == "interest_rate":
        return [f"The interest rate for the {product} is {v}% per annum, fixed for the loan term.",
                f"The {product} carries an annual interest rate of {v}%."]
    if category == "repayment":
        return [f"The maximum repayment tenure for the {product} is {v} months.",
                f"Borrowers may repay the {product} over up to {v} months."]
    if category == "documentation":
        return [f"Applicants for the {product} must submit the following documents: {', '.join(v)}.",
                f"Required documents for the {product} are {', '.join(v)}."]
    return [f"The {product} is secured against {v} as collateral.",
            f"Collateral required for the {product}: {v}."]

def make_example(rng):
    category = rng.choice(list(FIELDS))
    product = rng.choice(PRODUCTS)
    value = sample_value(rng, category)
    snippet = rng.choice(templates(category, value, product))
    gold = {"category": category, "product": product, FIELDS[category]: value}
    prompt = ("Extract the loan policy details as JSON.\n\n"
              f"Policy text: {snippet}\n\n"
              "Return only JSON with keys: category, product, and the one category-specific field.")
    return {"prompt": prompt, "completion": json.dumps(gold, ensure_ascii=False),
            "category": category, "product": product, "field": FIELDS[category], "value": value}

rng = random.Random(42)
for name, n in (("train.jsonl", 240), ("heldout.jsonl", 60)):
    rows = [make_example(rng) for _ in range(n)]
    with open(os.path.join(DATA, name), "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"wrote {name} ({n} rows)")

In [ ]:
import collections, json, os
def snippet_from_prompt(prompt):
    start = prompt.index("Policy text: ") + len("Policy text: ")
    return prompt[start:prompt.index("\n\nReturn", start)]
for name in ("train.jsonl", "heldout.jsonl"):
    rows = [json.loads(l) for l in open(os.path.join(DATA, name)) if l.strip()]
    named = 0
    text_to_products = {}
    for r in rows:
        c = json.loads(r["completion"])
        assert c["category"] == r["category"] and c["product"] == r["product"]
        assert c[r["field"]] == r["value"]
        named += r["product"] in r["prompt"]
        text_to_products.setdefault(snippet_from_prompt(r["prompt"]), set()).add(r["product"])
    contradictory = {t: ps for t, ps in text_to_products.items() if len(ps) > 1}
    assert named == len(rows), f"{name}: only {named}/{len(rows)} prompts name their gold product"
    assert not contradictory, f"{name}: {len(contradictory)} policy texts map to multiple products"
    print(name, len(rows), dict(collections.Counter(r["category"] for r in rows)),
          f"prompts naming gold product: {named}/{len(rows)}, contradictory texts: 0 OK")


## 2. LoRA training

Qwen2.5-0.5B-Instruct, r=16, alpha=32, dropout 0.05, all attention + MLP projections, lr 2e-4, effective batch 16, 3 epochs, prompt tokens masked. 240 examples = 45 optimizer steps — a short run (see README for the quota note).

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

assert torch.cuda.is_available(), "No GPU in this kernel session -- enable GPU and re-run."
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"))
model.print_trainable_parameters()

rows = [json.loads(l) for l in open(os.path.join(DATA, "train.jsonl")) if l.strip()]

class SFTDataset(Dataset):
    def __init__(self, rows):
        self.examples = []
        for row in rows:
            prompt = row["prompt"] + "\n"
            full = prompt + row["completion"] + tokenizer.eos_token
            p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            f_ids = tokenizer(full, add_special_tokens=False)["input_ids"][:256]
            labels = ([-100] * len(p_ids) + f_ids[len(p_ids):])[:256]
            self.examples.append({"input_ids": f_ids, "labels": labels})
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        return self.examples[i]

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [tokenizer.pad_token_id] * pad)
        labels.append(b["labels"] + [-100] * pad)
        attn.append([1] * len(b["input_ids"]) + [0] * pad)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}

trainer = Trainer(model=model, train_dataset=SFTDataset(rows), data_collator=collate,
                  args=TrainingArguments(output_dir=OUT, num_train_epochs=3.0,
                      per_device_train_batch_size=4, gradient_accumulation_steps=4,
                      learning_rate=2e-4, logging_steps=5, save_strategy="no",
                      bf16=True, report_to=[]))
trainer.train()
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print("saved LoRA adapter to", OUT)

## 3. Base vs LoRA evaluation

Greedy generation on the 60 held-out examples, scored on JSON validity, category, product, and the category-specific field.

In [ ]:
import json, os, re
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

FIELDS = {"eligibility": "min_credit_score", "interest_rate": "interest_rate_percent",
          "repayment": "max_tenure_months", "documentation": "required_documents",
          "collateral": "collateral_type"}

def extract_json(text):
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def score(gold, preds):
    n = len(gold)
    c = {"json_valid": 0, "category": 0, "product": 0, "field": 0, "overall": 0}
    for g, text in zip(gold, preds):
        p = extract_json(text)
        ok = [False, False, False]
        if p is not None:
            c["json_valid"] += 1
            ok[0] = p.get("category") == g["category"]
            ok[1] = p.get("product") == g["product"]
            f = FIELDS[g["category"]]
            ok[2] = f in p and (float(p[f]) == float(g["value"]) if isinstance(g["value"], (int, float)) else p[f] == g["value"])
        for k, v in zip(("category", "product", "field"), ok):
            c[k] += v
        c["overall"] += all(ok)
    per_product = {}
    for prod in sorted({g["product"] for g in gold}):
        idx = [i for i, g in enumerate(gold) if g["product"] == prod]
        per_product[prod] = {
            "n": len(idx),
            "product_acc": round(sum(1 for i in idx if extract_json(preds[i]) and extract_json(preds[i]).get("product") == gold[i]["product"]) / len(idx), 3),
            "overall": round(sum(1 for i in idx if extract_json(preds[i]) and extract_json(preds[i]).get("category") == gold[i]["category"] and extract_json(preds[i]).get("product") == gold[i]["product"] and FIELDS[gold[i]["category"]] in extract_json(preds[i]) and (float(extract_json(preds[i])[FIELDS[gold[i]["category"]]]) == float(gold[i]["value"]) if isinstance(gold[i]["value"], (int, float)) else extract_json(preds[i])[FIELDS[gold[i]["category"]]] == gold[i]["value"])) / len(idx), 3),
        }
    out = {k: round(v / n, 3) for k, v in c.items()}
    out["per_product"] = per_product
    return out

def generate(model, tok, gold):
    preds = []
    for row in gold:
        ids = tok(row["prompt"] + "\n", return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = model.generate(**ids, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        preds.append(tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
    return preds

gold = [json.loads(l) for l in open(os.path.join(DATA, "heldout.jsonl")) if l.strip()]
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).cuda().eval()
base_preds = generate(base, tok, gold)
base_score_tmp = score(gold, base_preds)
print("BASE:", base_score_tmp)
print("BASE per-product:", base_score_tmp["per_product"])
del base
lora = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16), OUT).cuda().eval()
lora_preds = generate(lora, tok, gold)
lora_score = score(gold, lora_preds)
print("LORA:", lora_score)
print("LORA per-product:", lora_score["per_product"])
base_score = score(gold, base_preds)
ok = lora_score["overall"] >= 0.80 and lora_score["json_valid"] == 1.0 and lora_score["overall"] >= base_score["overall"] + 0.20
print("PASS" if ok else "FAIL",
      "-- bar: LoRA overall >= 0.80, json_valid == 1.00, and LoRA >= base + 0.20")

## Success bar

LoRA `overall` >= 0.80, `json_valid` == 1.00, and LoRA >= base + 0.20. If it fails, iterate (more epochs/data) — do not claim the fine-tune worked. The adapter is saved to `/kaggle/working/outputs/lora-policy` in this kernel's output.